# KV-cache attention kernels, step by step

This notebook follows [Part II: KV Caches and Attention Kernels](https://g-u-n.github.io/blogs/scaling-long-context-kv-cache.html). We will run each implementation step independently: full-sequence MHA/GQA/MQA, backward, grouped decode, split-KV, Paged Attention, and absorbed MLA.

Run the cells in order. Each section compares the Triton result with a small PyTorch reference, so a failure stays attached to the idea that introduced it.

## 0. Connect a GPU and prepare Triton

Colab should request a GPU runtime from the notebook metadata. If CUDA is unavailable, select **Runtime → Change runtime type → GPU**. We use fp16 by default because it works on T4 as well as newer Colab GPUs.

In [ ]:
import importlib.util
import math
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU is attached. Select Runtime → Change runtime type → GPU.")

try:
    import triton
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "triton"], check=True)
    import triton

DEVICE = "cuda"
DTYPE = torch.float16
torch.manual_seed(0)

print("PyTorch:", torch.__version__)
print("Triton:", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("dtype used below:", DTYPE)
print("bf16 supported:", torch.cuda.is_bf16_supported())

We import the implementation from the repository rather than copying kernels into this notebook. That keeps the blog, the downloadable Python file, and this Colab on one source of truth.

In [ ]:
repo = Path("/content/G-U-N.github.io")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/G-U-N/G-U-N.github.io.git", str(repo)], check=True)

module_path = repo / "blogs/code/kv_cache_attention.py"
spec = importlib.util.spec_from_file_location("kv_cache_attention", module_path)
kva = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = kva
spec.loader.exec_module(kva)
print("Loaded:", module_path)

In [ ]:
def check_close(name, got, reference, *, atol=3e-2, rtol=3e-2):
    torch.testing.assert_close(got, reference, atol=atol, rtol=rtol)
    max_error = (got.float() - reference.float()).abs().max().item()
    print(f"{name:<24} shape={tuple(got.shape)!s:<22} max error={max_error:.3e}")

## 1. MHA → GQA → MQA changes only the KV-head mapping

We keep eight query heads and change the number of KV heads from 8 to 2 to 1. The mapping is $h_{kv}=\lfloor h_q/G\rfloor$, where $G=H_q/H_{kv}$. Each query head still computes its own attention distribution; only the K/V head it reads changes.

In [ ]:
B, H_Q, M, N, D_QK, D_V = 1, 8, 17, 23, 64, 48
SCALE = 0.125
q = torch.randn((B, H_Q, M, D_QK), device=DEVICE, dtype=DTYPE)

for h_kv, label in ((8, "MHA"), (2, "GQA"), (1, "MQA")):
    k = torch.randn((B, h_kv, N, D_QK), device=DEVICE, dtype=DTYPE)
    v = torch.randn((B, h_kv, N, D_V), device=DEVICE, dtype=DTYPE)
    group_size = H_Q // h_kv
    mapping = [head // group_size for head in range(H_Q)]

    got, got_lse = kva.flash_gqa_forward(q, k, v, scale=SCALE, causal=True)
    ref, ref_lse = kva.attention_reference(q, k, v, scale=SCALE, causal=True)

    print(f"\n{label}: H_q={H_Q}, H_kv={h_kv}, G={group_size}, mapping={mapping}")
    check_close(f"{label} output", got, ref)
    check_close(f"{label} LSE", got_lse, ref_lse)

## 2. Backward adds a reduction across shared query heads

The attention derivative is unchanged from Part I. GQA adds one ownership issue: every query head first produces a partial $dK$ and $dV$, then the $G$ partials that share one KV head are summed. The final gradient shapes therefore match K and V, not Q.

In [ ]:
B, H_Q, M, N, D_QK, D_V = 1, 4, 17, 23, 64, 48
GRAD_ATOL = 8e-2 if DTYPE == torch.bfloat16 else 5e-2

for h_kv, label, causal in ((4, "MHA", False), (2, "GQA", True), (1, "MQA", True)):
    q0 = torch.randn((B, H_Q, M, D_QK), device=DEVICE, dtype=DTYPE) * 0.4
    k0 = torch.randn((B, h_kv, N, D_QK), device=DEVICE, dtype=DTYPE) * 0.4
    v0 = torch.randn((B, h_kv, N, D_V), device=DEVICE, dtype=DTYPE) * 0.4
    do = torch.randn((B, H_Q, M, D_V), device=DEVICE, dtype=DTYPE) * 0.4

    q_ref, k_ref, v_ref = [x.detach().clone().requires_grad_(True) for x in (q0, k0, v0)]
    out_ref, _ = kva.attention_reference(q_ref, k_ref, v_ref, scale=SCALE, causal=causal)
    grads_ref = torch.autograd.grad(out_ref, (q_ref, k_ref, v_ref), do)

    q_tri, k_tri, v_tri = [x.detach().clone().requires_grad_(True) for x in (q0, k0, v0)]
    out_tri = kva.flash_gqa_attention(q_tri, k_tri, v_tri, scale=SCALE, causal=causal)
    grads_tri = torch.autograd.grad(out_tri, (q_tri, k_tri, v_tri), do)

    print(f"\n{label}: dQ={tuple(grads_tri[0].shape)}, dK={tuple(grads_tri[1].shape)}, dV={tuple(grads_tri[2].shape)}")
    check_close(f"{label} output", out_tri, out_ref)
    for name, got, ref in zip(("dQ", "dK", "dV"), grads_tri, grads_ref):
        check_close(f"{label} {name}", got, ref, atol=GRAD_ATOL, rtol=GRAD_ATOL)

## 3. Grouped-head decode reuses one KV tile across query heads

Decode has one new query token, so the query-head rows become the matrix-multiplication M dimension. One program handles a tile of query heads from the same KV group and streams the cache once for that tile.

In [ ]:
B, H_Q, H_KV, N, D_QK, D_V = 1, 8, 2, 257, 64, 48
q_decode = torch.randn((B, H_Q, D_QK), device=DEVICE, dtype=DTYPE)
k_cache = torch.randn((B, H_KV, N, D_QK), device=DEVICE, dtype=DTYPE)
v_cache = torch.randn((B, H_KV, N, D_V), device=DEVICE, dtype=DTYPE)
seq_lens = torch.tensor([219], device=DEVICE, dtype=torch.int32)

decode_ref, decode_ref_lse = kva.decode_reference(
    q_decode, k_cache, v_cache, seq_lens, scale=SCALE
)
decode_out, decode_lse = kva.grouped_decode(
    q_decode, k_cache, v_cache, seq_lens, scale=SCALE, num_splits=1
)

print(f"G={H_Q // H_KV}: each KV head is shared by {H_Q // H_KV} query heads")
check_close("grouped decode output", decode_out, decode_ref)
check_close("grouped decode LSE", decode_lse, decode_ref_lse)

## 4. Flash-Decoding splits the long KV axis

Each split computes a normalized partial output $O_s$ and a local $L_s=\operatorname{LSE}_s$. The combine kernel reconstructs the global result as $O=\sum_s e^{L_s-L}O_s$, where $L=\log\sum_s e^{L_s}$. Splitting changes the schedule, not the attention result.

In [ ]:
split_out, split_lse = kva.grouped_decode(
    q_decode, k_cache, v_cache, seq_lens, scale=SCALE, num_splits=4
)

check_close("split-KV vs reference", split_out, decode_ref)
check_close("split LSE vs reference", split_lse, decode_ref_lse)
check_close("split vs unsplit", split_out, decode_out)

## 5. Paged Attention changes address translation

The attention loop still processes logical token positions in order. A block table now maps each logical page to a physical cache page before K and V are loaded. We deliberately shuffle the physical pages below so the mapping is visible.

In [ ]:
PAGE_SIZE = 16
PHYSICAL_PAGES = 24
k_pages, block_table = kva._pack_pages(k_cache, PAGE_SIZE, PHYSICAL_PAGES)
v_pages = torch.zeros(
    (PHYSICAL_PAGES, PAGE_SIZE, H_KV, D_V), device=DEVICE, dtype=DTYPE
)

for batch in range(B):
    for logical_page in range(block_table.shape[1]):
        start = logical_page * PAGE_SIZE
        end = min(start + PAGE_SIZE, N)
        physical_page = int(block_table[batch, logical_page].item())
        v_pages[physical_page, : end - start] = v_cache[batch, :, start:end].transpose(0, 1)

print("logical page → physical page:", block_table[0].cpu().tolist())
paged_out, paged_lse = kva.paged_grouped_decode(
    q_decode, k_pages, v_pages, block_table, seq_lens, scale=SCALE, num_splits=4
)
check_close("paged output", paged_out, decode_ref)
check_close("paged LSE", paged_lse, decode_ref_lse)

## 6. MLA expands for training and absorbs for decode

For the synthetic example below, training materializes a content key and value for every head. Decode instead compares each absorbed query directly with the shared latent $c^{KV}$, adds the shared rotary-key score, and returns a weighted latent. We first verify the matrix algebra in PyTorch, then run the Triton decode kernel at the representative $512+64/512$ widths.

In [ ]:
B, H_Q, N = 1, 8, 129
D_HEAD, D_C, D_R, D_V_HEAD = 32, 512, 64, 32
mla_scale = 1.0 / math.sqrt(D_HEAD + D_R)

q_content = torch.randn((B, H_Q, D_HEAD), device=DEVICE, dtype=DTYPE)
q_rope = torch.randn((B, H_Q, D_R), device=DEVICE, dtype=DTYPE)
c_kv = torch.randn((B, N, D_C), device=DEVICE, dtype=DTYPE)
k_rope = torch.randn((B, N, D_R), device=DEVICE, dtype=DTYPE)
w_uk = torch.randn((H_Q, D_HEAD, D_C), device=DEVICE, dtype=DTYPE) / math.sqrt(D_C)
w_uv = torch.randn((H_Q, D_V_HEAD, D_C), device=DEVICE, dtype=DTYPE) / math.sqrt(D_C)

unabsorbed = kva.mla_unabsorbed_reference(
    q_content, q_rope, c_kv, k_rope, w_uk, w_uv, scale=mla_scale
)
q_absorbed, latent_ref, absorbed = kva.mla_absorbed_reference(
    q_content, q_rope, c_kv, k_rope, w_uk, w_uv, scale=mla_scale
)

print("q_content:", tuple(q_content.shape))
print("q_absorbed:", tuple(q_absorbed.shape))
print("shared c_kv:", tuple(c_kv.shape))
print("shared k_rope:", tuple(k_rope.shape))
check_close("absorbed algebra", absorbed, unabsorbed, atol=8e-3, rtol=8e-3)

In [ ]:
mla_seq_lens = torch.tensor([N], device=DEVICE, dtype=torch.int32)
latent, mla_lse = kva.mla_decode(
    q_absorbed.to(DTYPE),
    q_rope,
    c_kv,
    k_rope,
    mla_seq_lens,
    scale=mla_scale,
    num_splits=2,
)
kernel_out = torch.einsum("bhc,hvc->bhv", latent.float(), w_uv.float())

check_close("MLA latent", latent, latent_ref.to(DTYPE), atol=4e-2, rtol=4e-2)
check_close("MLA projected output", kernel_out, absorbed, atol=4e-2, rtol=4e-2)
print("MLA LSE shape:", tuple(mla_lse.shape))

## Done

We changed one systems axis at a time and checked the result after every change. On a GPU with bf16 support, set `DTYPE = torch.bfloat16` in the setup cell and rerun the notebook to repeat the same steps in bf16.